In [1]:
import numpy as np
import pandas as pd
import pulp
from sensitivity import heuristic3
import json
from pathlib import Path
import torch

In [2]:
tradeoff_map = np.array([
    [0.024842333333333335,     139],
    [0.022418333333333333,     285],
    [0.005855666666666667,   34405],
    [0.008697000000000000,   68741],
    [0.001205666666666667,  137777],
    [0.001235666666666667,  275393],
    [0.001804333333333333,  550753],
])

In [3]:
sensitivity = [5.829618417354863, 5.829618417354863, 6.783432564382072]

In [11]:
selection = heuristic3(
    sensitivity=sensitivity,
    budget=100000,
    tradeoff_map=tradeoff_map,
)

print(selection)

[4, 4, 4]


In [18]:
jsonl_path = Path("./output-dir/3944ba_4/metrics.jsonl")

results = []

with open(jsonl_path, "r") as f:
    for line in f:
        row = json.loads(line)

        if row["map"] == "F1/Shanghai/Shanghai":
            continue

        runs = row["runs"]

        baseline_rmse = None
        arch8_rmse = None

        for run in runs:
            if run["label"] == "comb1":
                baseline_rmse = run["rmse"]
            else:
                # assumes the other entry is your arch8 run
                arch8_rmse = run["rmse"]

        results.append({
            "map": row["map"],
            "baseline_rmse": baseline_rmse,
            "arch8_rmse": arch8_rmse,
        })


df = pd.DataFrame(results)

        

df.loc[len(df)] = {
    "map": "MEAN",
    "baseline_rmse": df["baseline_rmse"].mean(),
    "arch8_rmse": df["arch8_rmse"].mean(),
            }

df

,map,baseline_rmse,arch8_rmse
0,F1/MexicoCity/MexicoCity,0.161500,0.167400
1,F1/Monza/Monza,0.107400,0.105300
2,F1/Nuerburgring/Nuerburgring,0.163000,0.162000
3,F1/Silverstone/Silverstone,0.136600,0.130300
4,F1/Sochi/Sochi,0.136000,0.136700
5,F1/Spa/Spa,0.127600,0.191300
6,MEAN,0.138683,0.148833


In [9]:


selection = [2, 4, 4]

# heuristic3 returns level indices, convert to arch numbers
track_arch = selection[0] + 1
left_arch = selection[1] + 1
heading_arch = selection[2] + 1

# --------------------------------------------------
# Arch1-7 parameter lookup table
# --------------------------------------------------

arch_sizes = {
    1: 139,
    2: 285,
    3: 34405,
    4: 68741,
    5: 137777,
    6: 275393,
    7: 550753,
}

baseline_sizes = {
    "track_width": arch_sizes[track_arch],
    "left_wall_dist": arch_sizes[left_arch],
    "heading_error": arch_sizes[heading_arch],
}

# --------------------------------------------------
# Arch8 models
# --------------------------------------------------

track_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/3944ba/track_width_arch8_trial495.pt"

left_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/3944ba/left_wall_dist_arch8_trial495.pt"

heading_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/3944ba/heading_error_arch8_trial495.pt"

def count_params(path):
    model = torch.jit.load(path, map_location="cpu")
    return sum(p.numel() for p in model.parameters())

arch8_sizes = {
    "track_width": count_params(track_path),
    "left_wall_dist": count_params(left_path),
    "heading_error": count_params(heading_path),
}

# --------------------------------------------------
# Build comparison table
# --------------------------------------------------

df = pd.DataFrame({
    "target": ["track_width", "left_wall_dist", "heading_error"],
    "baseline_arch": [track_arch, left_arch, heading_arch],
    "baseline_params": [
        baseline_sizes["track_width"],
        baseline_sizes["left_wall_dist"],
        baseline_sizes["heading_error"],
    ],
    "arch8_params": [
        arch8_sizes["track_width"],
        arch8_sizes["left_wall_dist"],
        arch8_sizes["heading_error"],
    ],
})

df["param_difference"] = (
    df["arch8_params"] - df["baseline_params"]
)

# Total row
df.loc[len(df)] = {
    "target": "TOTAL",
    "baseline_arch": "",
    "baseline_params": df["baseline_params"].sum(),
    "arch8_params": df["arch8_params"].sum(),
    "param_difference":
        df["arch8_params"].sum()
        - df["baseline_params"].sum(),
}

df

,target,baseline_arch,baseline_params,arch8_params,param_difference
0,track_width,3,34405,138129,103724
1,left_wall_dist,5,137777,14173,-123604
2,heading_error,5,137777,138081,304
3,TOTAL,,309959,290383,-19576
